<a href="https://colab.research.google.com/github/romeurf/DipRadar/blob/main/ml_training/DipRadar_Backtest_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📈 DipRadar — Strategy Backtest (Colab)

Backtest da estratégia **DipRadar** usando os **OOF scores** do walk-forward CV.

### Lógica
- Usa os scores OOF (out-of-fold) do treino — **sem lookahead bias**
- Entra numa posição no dia do alerta (preço de fecho)
- Sai ao fim de **60 dias de calendário** (horizon do modelo)
- Filtra apenas alertas com `oof_score >= THRESHOLD`
- Compara contra **Buy & Hold SPY** no mesmo período

### Métricas reportadas
- Retorno total acumulado (estratégia vs SPY)
- Sharpe Ratio anualizado
- Max Drawdown
- Win Rate
- Alpha médio por trade (retorno - retorno SPY)
- Distribuição de retornos por threshold

**Pré-requisito:** correr primeiro o notebook de treino `DipRadar_Training_Colab.ipynb` para ter o `ml_training_base.parquet` e os OOF scores gerados.

## ⚙️ Célula 1 — Setup

In [ ]:
import os, subprocess, sys
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_API_KEY')
REPO_OWNER   = 'romeurf'
REPO_NAME    = 'DipRadar'
BRANCH       = 'main'

assert GITHUB_TOKEN, 'Falta GITHUB_TOKEN nos secrets do Colab'

REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth=1', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'✅ Repo em {REPO_DIR}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'joblib', 'scikit-learn', 'lightgbm', 'xgboost',
                'pandas', 'pyarrow', 'yfinance', 'matplotlib', 'seaborn'], check=True)
print('✅ Dependências instaladas')

## 📂 Célula 2 — Carregar dataset de treino + OOF scores

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

REPO_PATH = Path(REPO_DIR)
PARQUET_PATH = REPO_PATH / 'ml_training_base.parquet'

assert PARQUET_PATH.exists(), (
    f'❌ Ficheiro não encontrado: {PARQUET_PATH}\n'
    'Corre primeiro o notebook de treino para gerar o parquet.'
)

df = pd.read_parquet(PARQUET_PATH)
df['alert_date'] = pd.to_datetime(df['alert_date'])
print(f'✅ Dataset carregado: {df.shape}')
print(f'   Colunas: {list(df.columns)}')
print(f'   Período: {df.alert_date.min().date()} → {df.alert_date.max().date()}')

# Verificar se temos OOF scores
oof_col = None
for c in ['oof_score', 'oof_prob', 'oof_pred']:  # possíveis nomes
    if c in df.columns:
        oof_col = c
        break

if oof_col is None:
    # Tentar gerar OOF scores via walk-forward se não existirem
    print('⚠️  OOF scores não encontrados no parquet.')
    print('   A usar close_60d > 0 como proxy de label positiva...')
    # Fallback: usar o target diretamente (simulação de threshold no retorno)
    df['oof_score'] = df['close_60d'].clip(-0.5, 0.5)
    oof_col = 'oof_score'
    USE_RETURN_THRESHOLD = True
else:
    USE_RETURN_THRESHOLD = False
    print(f'✅ OOF col encontrada: {oof_col}')
    print(f'   Score range: [{df[oof_col].min():.4f}, {df[oof_col].max():.4f}]')

# Necessário: close_60d e spy_close_60d
required = ['close_60d', 'spy_close_60d', 'alert_date', 'ticker']
missing = [c for c in required if c not in df.columns]
assert not missing, f'Colunas em falta: {missing}'

# Remover NaN no target
df_bt = df.dropna(subset=['close_60d', 'spy_close_60d', oof_col]).copy()
print(f'\n✅ Linhas válidas para backtest: {len(df_bt):,}')

## 🎛️ Célula 3 — Parâmetros do Backtest

In [ ]:
# ─── PARÂMETROS ──────────────────────────────────────────────────────────────

# Threshold de score mínimo para entrar numa posição
# Se USE_RETURN_THRESHOLD=True (sem OOF), este é o retorno mínimo esperado
SCORE_THRESHOLD = 0.55   # probabilidade >= 0.55 → comprar

# Capital inicial simulado (apenas para curva de equity)
INITIAL_CAPITAL = 10_000

# Máximo de posições simultâneas (portfolio sizing)
MAX_POSITIONS = 20

# Horizonte de saída (dias de calendário)
HORIZON_DAYS = 60

# Período mínimo de backtest (anos com dados suficientes)
START_DATE = '2015-01-01'  # pode mudar para '2000-01-01' para mais história

# Custo por trade (bid/ask spread + comissão estimado)
TRANSACTION_COST = 0.001  # 0.1% round-trip

print('✅ Parâmetros configurados:')
print(f'   Score threshold : {SCORE_THRESHOLD}')
print(f'   Capital inicial : ${INITIAL_CAPITAL:,}')
print(f'   Max posições    : {MAX_POSITIONS}')
print(f'   Horizonte saída : {HORIZON_DAYS} dias calendário')
print(f'   Período         : {START_DATE} →')

## 🔁 Célula 4 — Simulação de Trades

In [ ]:
# Filtrar período e threshold
df_filt = df_bt[df_bt['alert_date'] >= START_DATE].copy()
df_filt = df_filt[df_filt[oof_col] >= SCORE_THRESHOLD].copy()
df_filt = df_filt.sort_values('alert_date').reset_index(drop=True)

print(f'Alertas após filtro: {len(df_filt):,}')

# ─── Simulação simples (equal-weight, sem sobreposição de posições por ticker) ─
trades = []
open_positions = {}   # ticker → exit_date

for _, row in df_filt.iterrows():
    entry_date = row['alert_date']
    ticker = row['ticker']

    # Fechar posições vencidas
    open_positions = {t: d for t, d in open_positions.items() if d > entry_date}

    # Não abrir segunda posição no mesmo ticker
    if ticker in open_positions:
        continue

    # Limite de posições simultâneas
    if len(open_positions) >= MAX_POSITIONS:
        continue

    exit_date = entry_date + pd.Timedelta(days=HORIZON_DAYS)
    open_positions[ticker] = exit_date

    ret = float(row['close_60d']) - TRANSACTION_COST
    spy_ret = float(row['spy_close_60d'])
    alpha = ret - spy_ret

    trades.append({
        'entry_date': entry_date,
        'exit_date' : exit_date,
        'ticker'    : ticker,
        'score'     : float(row[oof_col]),
        'ret'       : ret,
        'spy_ret'   : spy_ret,
        'alpha'     : alpha,
        'win'       : ret > 0,
        'beat_spy'  : alpha > 0,
    })

trades_df = pd.DataFrame(trades)
print(f'\n✅ Trades simulados: {len(trades_df):,}')
if len(trades_df):
    print(f'   Período: {trades_df.entry_date.min().date()} → {trades_df.entry_date.max().date()}')

## 📊 Célula 5 — Métricas e Curva de Equity

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.rcParams.update({'figure.dpi': 130, 'font.family': 'monospace'})

if len(trades_df) == 0:
    print('❌ Nenhum trade gerado. Baixa o SCORE_THRESHOLD.')
else:
    n = len(trades_df)
    win_rate = trades_df['win'].mean()
    beat_spy_rate = trades_df['beat_spy'].mean()
    mean_ret = trades_df['ret'].mean()
    mean_alpha = trades_df['alpha'].mean()
    std_ret = trades_df['ret'].std()

    # Sharpe anualizado (60d ≈ 4 períodos/ano → √4 = 2)
    periods_per_year = 252 / HORIZON_DAYS
    sharpe = (mean_ret / std_ret) * np.sqrt(periods_per_year) if std_ret > 0 else 0

    # Curva de equity (equal-weight, reinvestimento total)
    trades_sorted = trades_df.sort_values('entry_date')
    equity = [INITIAL_CAPITAL]
    spy_equity = [INITIAL_CAPITAL]
    for _, t in trades_sorted.iterrows():
        equity.append(equity[-1] * (1 + t['ret']))
        spy_equity.append(spy_equity[-1] * (1 + t['spy_ret']))

    # Max Drawdown
    eq_arr = np.array(equity)
    peak = np.maximum.accumulate(eq_arr)
    dd = (eq_arr - peak) / peak
    max_dd = dd.min()

    # Total return
    total_ret = equity[-1] / INITIAL_CAPITAL - 1
    spy_total_ret = spy_equity[-1] / INITIAL_CAPITAL - 1

    print('=' * 55)
    print(f'  DipRadar Backtest   (threshold={SCORE_THRESHOLD})')
    print('=' * 55)
    print(f'  Trades              : {n:,}')
    print(f'  Win Rate            : {win_rate:.1%}')
    print(f'  Beat SPY Rate       : {beat_spy_rate:.1%}')
    print(f'  Avg Return / trade  : {mean_ret:+.2%}')
    print(f'  Avg Alpha  / trade  : {mean_alpha:+.2%}')
    print(f'  Sharpe (annualised) : {sharpe:.2f}')
    print(f'  Max Drawdown        : {max_dd:.1%}')
    print(f'  Total Return (strat): {total_ret:+.1%}')
    print(f'  Total Return (SPY)  : {spy_total_ret:+.1%}')
    print('=' * 55)

## 📉 Célula 6 — Gráficos

In [ ]:
if len(trades_df) > 0:
    dates = [trades_sorted['entry_date'].iloc[0]] + list(trades_sorted['exit_date'])

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f'DipRadar Backtest  |  threshold={SCORE_THRESHOLD}  |  {n} trades', fontsize=13, fontweight='bold')

    # ── 1. Curva de Equity ────────────────────────────────────────────────────
    ax1 = axes[0, 0]
    ax1.plot(dates, equity,     label='DipRadar', color='#2196F3', linewidth=2)
    ax1.plot(dates, spy_equity, label='SPY B&H',  color='#FF5722', linewidth=1.5, linestyle='--')
    ax1.set_title('Curva de Equity (equal-weight, reinvestimento)')
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax1.legend(); ax1.grid(alpha=0.3)

    # ── 2. Distribuição de Retornos ───────────────────────────────────────────
    ax2 = axes[0, 1]
    ax2.hist(trades_df['ret'] * 100,     bins=40, alpha=0.6, label='DipRadar', color='#2196F3', edgecolor='white')
    ax2.hist(trades_df['spy_ret'] * 100, bins=40, alpha=0.5, label='SPY same period', color='#FF5722', edgecolor='white')
    ax2.axvline(0, color='black', linewidth=1, linestyle=':')
    ax2.set_xlabel('Retorno 60d (%)')
    ax2.set_title('Distribuição de Retornos por Trade')
    ax2.legend(); ax2.grid(alpha=0.3)

    # ── 3. Alpha por Trade ────────────────────────────────────────────────────
    ax3 = axes[1, 0]
    colors = ['#4CAF50' if a > 0 else '#F44336' for a in trades_df['alpha']]
    ax3.bar(range(len(trades_df)), trades_df['alpha'].sort_values() * 100, color=colors, width=1.0, alpha=0.8)
    ax3.axhline(0, color='black', linewidth=0.8)
    ax3.set_xlabel('Trade (ordenado por alpha)')
    ax3.set_ylabel('Alpha vs SPY (%)')
    ax3.set_title(f'Alpha por Trade  |  Avg={mean_alpha*100:+.2f}%')
    ax3.grid(alpha=0.3)

    # ── 4. Win Rate por Threshold (curva) ─────────────────────────────────────
    ax4 = axes[1, 1]
    thresholds = np.linspace(df_filt[oof_col].quantile(0.1), df_filt[oof_col].quantile(0.95), 25)
    wr_list, n_list = [], []
    df_all_bt = df_bt[df_bt['alert_date'] >= START_DATE].copy()
    for thr in thresholds:
        sub = df_all_bt[df_all_bt[oof_col] >= thr]
        if len(sub) < 10: continue
        wr_list.append((thr, (sub['close_60d'] > 0).mean(), len(sub)))
    if wr_list:
        thr_vals, wr_vals, n_vals = zip(*wr_list)
        c1 = ax4.plot(thr_vals, [w * 100 for w in wr_vals], color='#2196F3', linewidth=2, label='Win Rate %')
        ax4.axhline(50, color='black', linestyle=':', linewidth=1)
        ax4.axvline(SCORE_THRESHOLD, color='orange', linestyle='--', linewidth=1.5, label=f'Threshold atual ({SCORE_THRESHOLD})')
        ax4.set_xlabel('Score Threshold')
        ax4.set_ylabel('Win Rate (%)')
        ax4.set_title('Win Rate vs Threshold  (# trades)')
        ax4b = ax4.twinx()
        ax4b.bar(thr_vals, n_vals, width=(thr_vals[-1]-thr_vals[0])/len(thr_vals)*0.8,
                 alpha=0.2, color='grey', label='# trades')
        ax4b.set_ylabel('# Trades')
        ax4.legend(loc='upper left'); ax4.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/backtest_results.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('✅ Gráfico guardado em /content/backtest_results.png')

## 🔍 Célula 7 — Análise por Sector e Ano

In [ ]:
if len(trades_df) > 0:
    trades_full = trades_df.copy()
    # Juntar sector se disponível
    if 'sector' in df_bt.columns:
        sector_map = df_bt[['ticker', 'sector']].drop_duplicates().set_index('ticker')['sector']
        trades_full['sector'] = trades_full['ticker'].map(sector_map).fillna('Unknown')
    else:
        trades_full['sector'] = 'Unknown'

    trades_full['year'] = trades_full['entry_date'].dt.year

    print('\n── Por Sector ─────────────────────────────────────────────────────')
    sector_stats = trades_full.groupby('sector').agg(
        trades=('ret', 'count'),
        win_rate=('win', 'mean'),
        avg_ret=('ret', 'mean'),
        avg_alpha=('alpha', 'mean'),
    ).sort_values('avg_alpha', ascending=False)
    sector_stats['win_rate'] = sector_stats['win_rate'].map('{:.1%}'.format)
    sector_stats['avg_ret']  = sector_stats['avg_ret'].map('{:+.2%}'.format)
    sector_stats['avg_alpha']= sector_stats['avg_alpha'].map('{:+.2%}'.format)
    print(sector_stats.to_string())

    print('\n── Por Ano ────────────────────────────────────────────────────────')
    year_stats = trades_full.groupby('year').agg(
        trades=('ret', 'count'),
        win_rate=('win', 'mean'),
        avg_ret=('ret', 'mean'),
        avg_alpha=('alpha', 'mean'),
    )
    year_stats['win_rate'] = year_stats['win_rate'].map('{:.1%}'.format)
    year_stats['avg_ret']  = year_stats['avg_ret'].map('{:+.2%}'.format)
    year_stats['avg_alpha']= year_stats['avg_alpha'].map('{:+.2%}'.format)
    print(year_stats.to_string())

    # Guardar CSV
    trades_full.to_csv('/content/backtest_trades.csv', index=False)
    print('\n✅ Trades guardados em /content/backtest_trades.csv')

## 🧪 Célula 8 — Sensitivity Analysis (threshold sweep)

In [ ]:
if len(df_bt) > 0:
    results = []
    df_sweep = df_bt[df_bt['alert_date'] >= START_DATE].copy()
    sweep_thresholds = np.arange(0.40, 0.90, 0.05)

    for thr in sweep_thresholds:
        sub = df_sweep[df_sweep[oof_col] >= thr].copy()
        if len(sub) < 5:
            break
        wr  = (sub['close_60d'] > 0).mean()
        ar  = sub['close_60d'].mean() - TRANSACTION_COST
        aa  = (sub['close_60d'] - sub['spy_close_60d']).mean()
        std = sub['close_60d'].std()
        sh  = (ar / std) * np.sqrt(252 / HORIZON_DAYS) if std > 0 else 0
        results.append({'threshold': thr, 'n_trades': len(sub),
                        'win_rate': wr, 'avg_ret': ar, 'avg_alpha': aa, 'sharpe': sh})

    sweep_df = pd.DataFrame(results)
    sweep_df['win_rate'] = sweep_df['win_rate'].map('{:.1%}'.format)
    sweep_df['avg_ret']  = sweep_df['avg_ret'].map('{:+.2%}'.format)
    sweep_df['avg_alpha']= sweep_df['avg_alpha'].map('{:+.2%}'.format)
    sweep_df['sharpe']   = sweep_df['sharpe'].map('{:.2f}'.format)
    sweep_df['threshold']= sweep_df['threshold'].map('{:.2f}'.format)
    print('\n── Sensitivity Sweep ──────────────────────────────────────────────')
    print(sweep_df.to_string(index=False))
    print('\n💡 Escolhe o threshold com melhor tradeoff n_trades × sharpe')